# Khare, Timol: determining equations in SageMath

The examples of K. Khare, M. G. Timol, *Determining equations for infinitesimal
transformation of second and third-order ODE using algorithm in open-source SageMath*, as
entries of the symmetry catalogue (`tests/symmetry_catalog.py`), shown as in
`notebooks/Catalogue_template.ipynb`: the equation, the dimension of its Lie algebra of
point symmetries (the rank of the Janet basis) next to the catalogue's, the staircase of the
Janet basis and, without arbitrary functions, the generators as vector fields.

Example 3 has an arbitrary function F(x): no symmetries for a generic F. The Janet basis
assumes some differential polynomials in F nonzero (`assumed_nonzero()`); where one of them
vanishes, e.g. for F = 1, there are more.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from sympy import Eq, symbols
from sympy.core.function import AppliedUndef

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # the catalogue lives in tests/
from tests.symmetry_catalog import CATALOG
from tests.test_symmetry_catalog import janet_basis

from delierium import ltf
from delierium.visualization import ode_solutions, staircase, vector_fields


def show(name, values=None):
    # the template for one catalogue entry: equation, rank, staircase, vector fields
    entry = next(e for e in CATALOG if e.name == name)
    indep, dep = entry.variables()
    for e in entry.parsed_equations():
        ltf(Eq(e, 0), dep, indep)
    jb = janet_basis(entry)
    print(f"{entry.source}\nrank: {jb.rank()}   catalogue: {entry.dimension}")
    if entry.note:
        print("note:", entry.note)
    staircase(jb, names={"H": entry.dependent[0]}, title=entry.name)
    plt.show()
    generators = entry.parsed_generators()
    arbitrary = set().union(*(e.atoms(AppliedUndef) for e in entry.parsed_equations())) - set(dep)
    if arbitrary:
        print("arbitrary functions", sorted(map(str, arbitrary)), "- no pictures of the generators")
    elif generators:
        coordinates = symbols(entry.independent + entry.dependent)
        curves = (
            ode_solutions(entry.parsed_equations()[0], dep[0], values=values)
            if entry.kind == "ode"
            else []
        )
        axes = (1, 2) if entry.kind == "odes" else (0, 1)
        vector_fields(generators, coordinates, curves, axes=axes, values=values)
        plt.show()

## Examples

### Example 1: the modified Emden equation

In [ ]:
show('Arrigo 2.19 (modified Emden equation)')

### Example 2: the Chazy equation

In [ ]:
show('Chazy equation')

### Example 3: y'' = F(x)/y**2

In [ ]:
show("Khare-Timol Example 3: y'' = F(x)/y**2")